In [ ]:
import tensorflow as tf
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    print(f"Au fost detectate {len(gpus)} GPU-uri active:")
    for i, gpu in enumerate(gpus):
        print(f" Placa {i}: {gpu.name}")
else:
    print("Niciun GPU detectat. Sistemul rulează lent pe CPU.")


In [ ]:

strategy = tf.distribute.MirroredStrategy()
print(f"Numărul de unități grafice sincronizate: {strategy.num_replicas_in_sync}")

In [ ]:
import keras
import tensorflow as tf
import numpy as np
import os
from tensorflow.keras.preprocessing.image import ImageDataGenerator


train_dir = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/train'
test_dir  = '/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test'

print("Versiune TensorFlow: ", tf.__version__)
print("Versiune Keras: ", keras.__version__)

img_size = 48
batch_size = 16
epoci = 50
num_classes = 7  # RAF-DB are 7 clase de bază

print("\nClase găsite:")
print(os.listdir(train_dir))

datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.15,
    width_shift_range=0.1,
    height_shift_range=0.1,
    horizontal_flip=True,
    fill_mode='nearest',
    validation_split=0.20
)

test_datagen = ImageDataGenerator(rescale=1./255)

print("Setul de antrenare (80%):")
train_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=True,
    subset='training'
)

print("Setul de validare (20%):")
val_generator = datagen.flow_from_directory(
    train_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False,
    subset='validation'
)

print("Setul de testare:")
test_generator = test_datagen.flow_from_directory(
    test_dir,
    target_size=(img_size, img_size),
    color_mode="grayscale",
    batch_size=batch_size,
    class_mode='categorical',
    shuffle=False
)

In [ ]:
import random
import matplotlib.pyplot as plt
import numpy as np
from tensorflow.keras.preprocessing.image import load_img, img_to_array

clase_disponibile = os.listdir(train_dir)
clasa_random = random.choice(clase_disponibile)
folder_clasa = os.path.join(train_dir, clasa_random)

imagini_disponibile = os.listdir(folder_clasa)
imagine_random = random.choice(imagini_disponibile)
cale_imagine = os.path.join(folder_clasa, imagine_random)

img = load_img(cale_imagine, target_size=(img_size, img_size), color_mode="grayscale")
img_originala = img_to_array(img) / 255.0

img_augmentata = datagen.random_transform(img_originala)
img_augmentata = np.clip(img_augmentata, 0, 1)

plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.title("Original")
plt.imshow(img_originala.reshape(img_size, img_size), cmap='gray', vmin=0, vmax=1)
plt.axis('off')

plt.subplot(1, 2, 2)
plt.title("Augmentată")
plt.imshow(img_augmentata.reshape(img_size, img_size), cmap='gray', vmin=0, vmax=1)
plt.axis('off')

plt.tight_layout()
plt.show()

In [ ]:
#implementarea retelei de tip v-cnn (Dogaru & Dogaru, 2023)
from keras.models import Sequential
from keras.layers import Conv2D, MaxPooling2D, Flatten, Dense, Dropout, Input,Activation, BatchNormalization
from keras.optimizers import Adam 

def creare_v_cnn_model(input_shape, num_classes, flat=1, fil=[64,128,256], nl=[1,1,1], hid=[128]):
    model= Sequential()
    csize= 3; stri=2;
    psiz=2
    drop1=0.3

    model.add(Conv2D(fil[0],padding='same', kernel_size=(csize,csize),input_shape=input_shape))
    model.add(Activation('relu'))
    
    for i in range(nl[0]):
        model.add(Conv2D(fil[0], padding='same', kernel_size=(csize, csize)))
        model.add(Activation('relu'))

    model.add(BatchNormalization())
    model.add(MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding='same'))
    model.add(Dropout(drop1))

    for layer in range(1, len(fil)):
        for j in range(nl[layer]):
            model.add(Conv2D(fil[layer], padding='same', kernel_size=(csize, csize)))
            model.add(Activation('relu'))
            
        model.add(BatchNormalization())
        model.add(MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding='same'))
        model.add(Dropout(drop1))

    if flat == 1:
        model.add(Flatten())
    else:
        model.add(GlobalAveragePooling2D())

    for h_size in hid:
        model.add(Dense(h_size, activation='relu'))
        model.add(Dropout(drop1))
    
    model.add(Dense(num_classes, activation='softmax'))
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4), loss='categorical_crossentropy', metrics=['accuracy'])
    return model

print("Construim arhitectura V-CNN...")
with strategy.scope():
    model = creare_v_cnn_model(input_shape=(48, 48, 1), num_classes=7)

model.summary()

In [ ]:
#implementarea retelei de tip VRES-CNN (Dogaru & Dogaru, 2023)
import numpy as np
import tensorflow as tf
from keras.layers import (Input, SeparableConv2D, Conv2D, Activation,
                           BatchNormalization, Add, MaxPooling2D,
                           GlobalAveragePooling2D, Flatten, Dense, Dropout)
from keras.models import Model

def svcnn_resblock(x, filters, nl, resid=True, separ=True):
    csize = 3; stri = 2; psiz = 3; pad = 'same'; drop1 = 0.25

    def conv_bn_relu(inp):
        if separ:
            out = SeparableConv2D(filters, padding=pad, kernel_size=(csize, csize))(inp)
        else:
            out = Conv2D(filters, padding=pad, kernel_size=(csize, csize))(inp)
        out = BatchNormalization()(out)
        out = Activation('relu')(out)
        return out

    y = conv_bn_relu(x)
    for _ in range(nl):
        y = conv_bn_relu(y)

    if resid:
        if x.shape[-1] != filters:
            shortcut = Conv2D(filters, kernel_size=1, padding=pad)(x)
            shortcut = BatchNormalization()(shortcut)
        else:
            shortcut = x
        out = Add()([shortcut, y])
    else:
        out = y

    out = MaxPooling2D(pool_size=(psiz, psiz), strides=(stri, stri), padding=pad)(out)
    out = Dropout(drop1)(out)
    return out

def create_vres_cnn_ckplus(input_shape=(48, 48, 1), num_classes=7,
                         flat=0, fil=[64, 128, 256], nl=[3, 2, 2],
                         hid=[256], resid=True, separ=True):
    inputs = Input(shape=input_shape)
    t = svcnn_resblock(inputs, fil[0], nl[0], resid, separ)
    for layer in range(1, len(fil)):
        t = svcnn_resblock(t, fil[layer], nl[layer], resid, separ)

    if flat == 1:
        t = Flatten()(t)
    else:
        t = GlobalAveragePooling2D()(t)

    for units in hid:
        t = Dense(units, activation='relu')(t)
        t = Dropout(0.3)(t)

    t = Dropout(0.4)(t)
    outputs = Dense(num_classes, activation='softmax')(t)
    model = Model(inputs, outputs)

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=5e-4),
        loss=tf.keras.losses.CategoricalCrossentropy(label_smoothing=0.1),
        metrics=['accuracy']
    )
    return model

print("Construim arhitectura VRES-CNN...")
with strategy.scope():
    model = create_vres_cnn_ckplus(
        input_shape=(48, 48, 1),
        num_classes=7,
        flat=0,
        fil=[64, 128, 256],
        nl=[3, 2, 2],
        hid=[256],
        resid=True,
        separ=True
    )

model.summary()

In [ ]:
from tensorflow.keras.applications import MobileNet
from tensorflow.keras.layers import Input, Rescaling, Dense, Dropout, Flatten, BatchNormalization, Activation, GlobalAveragePooling2D
from tensorflow.keras.models import Model

def create_mobilenet_ckplus(input_shape=(48, 48, 1), num_classes=7):
    img_input = Input(shape=input_shape)
    
    x = tf.keras.layers.Concatenate()([img_input, img_input, img_input])
    
    base_model = MobileNet(
        input_shape=(48, 48, 3),
        weights='imagenet',
        include_top=False
    )
    base_model.trainable = True
    
    x = base_model(x)
    x = GlobalAveragePooling2D()(x)
    x = Dropout(0.5)(x)
    x = Dense(256, activation='relu')(x)
    x = Dropout(0.3)(x)
    outputs = Dense(num_classes, activation='softmax')(x)
    
    model = Model(inputs=img_input, outputs=outputs)
    return model

with strategy.scope():
    model = create_mobilenet_ckplus(input_shape=(48, 48, 1), num_classes=num_classes)
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-4),
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )

model.summary()

In [ ]:
from keras.callbacks import ModelCheckpoint, EarlyStopping
checkpoint=ModelCheckpoint(
    "mobilenet_raf-db_best_model.keras",
    monitor='val_accuracy', 
    verbose=1, 
    save_best_only=True, 
    mode='max'
)

early_stopping = EarlyStopping(
    monitor='val_accuracy', 
    patience=15,               
    restore_best_weights=True,
    verbose=1
)


In [ ]:
import time

t_start_train = time.time()
istoric_antrenare = model.fit(
    train_generator,
    epochs=epoci,
    validation_data=val_generator,
    callbacks=[checkpoint, early_stopping]
)
t_end_train = time.time()

t_start_test = time.time()
score = model.evaluate(test_generator, verbose=0)  
t_end_test = time.time()

nr_total_poze = test_generator.samples

timp_antrenament_minute = int(t_end_train - t_start_train) / 60
timp_testare_secunde = t_end_test - t_start_test
latenta_ms = 1000 * timp_testare_secunde / nr_total_poze

epoci_rulate = len(istoric_antrenare.history['loss'])

print('RAPORT DE PERFORMANȚĂ')
print(f'Timp total antrenament ({epoci_rulate} epoci): {timp_antrenament_minute:.2f} minute')
print(f'Numărul total de parametri: {model.count_params()}')
print(f'Acuratețea finală pe setul de test: {score[1]*100:.2f} %')
print(f'Timpul total de predicție la test: {timp_testare_secunde:.4f} secunde')
print(f'Latența pe GPU (per cadru): {latenta_ms:.4f} ms')

In [ ]:
import numpy as np
import seaborn as sns
from sklearn.metrics import confusion_matrix, classification_report
import matplotlib.pyplot as plt

print("Calculăm predicțiile pe setul de test...")
y_pred_probs = model.predict(test_generator, verbose=1)
y_pred = np.argmax(y_pred_probs, axis=1)
y_true = test_generator.classes

etichete = ['surprise', 'fear', 'disgust', 'happiness', 'sadness', 'anger', 'neutral']

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='PuRd',
            xticklabels=etichete, yticklabels=etichete)
plt.title('Matricea de Confuzie - RAF-DB')
plt.ylabel('Emoție Reală')
plt.xlabel('Emoție Prezisă')
plt.show()

print("\nRaport de Clasificare Detaliat:\n")
print(classification_report(y_true, y_pred, target_names=etichete, zero_division=0))

In [ ]:
import matplotlib.pyplot as plt


acc = istoric_antrenare.history['accuracy']
val_acc = istoric_antrenare.history['val_accuracy']
loss = istoric_antrenare.history['loss']
val_loss = istoric_antrenare.history['val_loss']

epoci_rulate = range(1, len(acc) + 1)


plt.figure(figsize=(14, 5))


plt.subplot(1, 2, 1)
plt.plot(epoci_rulate, acc, 'b-', linewidth=2, label='Antrenament')
plt.plot(epoci_rulate, val_acc, 'r-', linewidth=2, label='Validare')
plt.title('Evoluția Acurateții mobilenet raf-db', fontsize=14)
plt.xlabel('Număr Epoci', fontsize=12)
plt.ylabel('Acuratețe', fontsize=12)
plt.legend(loc='lower right')
plt.grid(True, linestyle='--', alpha=0.7)


plt.subplot(1, 2, 2)
plt.plot(epoci_rulate, loss, 'b-', linewidth=2, label='Eroare Antrenament')
plt.plot(epoci_rulate, val_loss, 'r-', linewidth=2, label='Eroare Validare')
plt.title('Scăderea Erorii (Loss) mobilenet raf-db', fontsize=14)
plt.xlabel('Număr Epoci', fontsize=12)
plt.ylabel('Valoare Eroare', fontsize=12)
plt.legend(loc='upper right')
plt.grid(True, linestyle='--', alpha=0.7)


plt.tight_layout()
plt.show()

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.preprocessing import image

def testeaza_cu_legenda(cale_imagine):
    
    etichete_en = ['surprise', 'fear', 'disgust', 'happiness', 'sadness', 'anger', 'neutral']
    
    clase_ro = {
        'surprise': 'SURPRIZĂ',
        'fear': 'FRICĂ',
        'disgust': 'DEZGUST',
        'happiness': 'FERICIRE',
        'sadness': 'TRISTEȚE',
        'anger': 'FURIE',
        'neutral': 'NEUTRU'
    }
    
    nume_ro_scurt = ['Surpriză', 'Frică', 'Dezgust', 'Fericire', 'Tristețe', 'Furie', 'Neutru']

    try:
        img_pt_model = image.load_img(cale_imagine, target_size=(48, 48), color_mode="grayscale")
    except Exception as e:
        print(f"Eroare la încărcarea imaginii: {e}")
        return

    img_array = image.img_to_array(img_pt_model)
    img_array = np.expand_dims(img_array, axis=0)
    img_array /= 255.0

    predictii_brute = model.predict(img_array)[0]
    index_maxim = np.argmax(predictii_brute)

    emotie_w_en = etichete_en[index_maxim]
    emotie_w_ro = clase_ro[emotie_w_en]
    procent_w = predictii_brute[index_maxim] * 100

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

    img_orig = cv2.imread(cale_imagine)
    if img_orig is None:
        print("Eroare: Nu am putut citi imaginea originală pentru afișare!")
        return
    img_orig = cv2.cvtColor(img_orig, cv2.COLOR_BGR2RGB)

    ax1.imshow(img_orig)
    ax1.axis('off')

    culoare_titlu = 'green' if emotie_w_en == 'happiness' else 'red' if emotie_w_en in ['anger', 'fear'] else 'darkblue'

    ax1.set_title(f"Modelul prezice:\n{emotie_w_ro} ({procent_w:.2f}%)",
                  fontsize=18, color=culoare_titlu, fontweight='bold', pad=20)

    y_pos = np.arange(len(nume_ro_scurt))
    procente_afisare = predictii_brute * 100

    bare = ax2.barh(y_pos, procente_afisare, align='center', color='skyblue', alpha=0.8)

    bare[index_maxim].set_color('deeppink')
    bare[index_maxim].set_alpha(1.0)

    ax2.set_yticks(y_pos)
    ax2.set_yticklabels(nume_ro_scurt, fontsize=12)
    ax2.invert_yaxis()
    ax2.set_xlabel('Probabilitate (%)', fontsize=12)
    ax2.set_title('Distribuția detaliată a probabilităților', fontsize=14, pad=15)

    for i, bar in enumerate(bare):
        latime = bar.get_width()
        ax2.text(latime + 1, bar.get_y() + bar.get_height()/2,
                 f'{procente_afisare[i]:.1f}%',
                 va='center', fontsize=10, fontweight='bold' if i == index_maxim else 'normal')

    ax2.set_xlim(0, 110)
    ax2.grid(axis='x', linestyle='--', alpha=0.5)
    plt.tight_layout()
    plt.show()

# o poza din RAF-DB test
testeaza_cu_legenda('/kaggle/input/datasets/shuvoalok/raf-db-dataset/DATASET/test/5/test_0005_aligned.jpg')


In [ ]:
from tensorflow.keras.models import load_model

cale_model = '/kaggle/input/datasets/mindrocrobert/modelulmeu/v_cnn_best_model (1).keras'
model = load_model(cale_model)

print("✅ SUCCES: Modelul este acum încărcat în memorie!")